In [1]:

# Analysis Plan:
# ==========================================
# Research Objective: Test if the N-stable GEV statistics of L(s,λ) result from 
# N-stable GEV statistics of its even-Ω and odd-Ω components.
#
# Steps:
# 1. Implement the Liouville function L(s,λ) and its even/odd Ω decomposition
# 2. Generate time series for D(t;N), D_even(t;N), D_odd(t;N) at N=10^5 and 10^6
# - Use t ∈ [5000, 25000] with 5000 points
# 3. Compute log|D| for each series
# 4. Fit GEV distributions to block maxima (100 blocks) for each series
# 5. Extract shape parameter ξ with 95% confidence intervals
# 6. Test N-stability by comparing ξ at N=10^5 vs N=10^6 (check CI overlap)
# 7. Determine if component stability explains total stability

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*80)
print("ANALYSIS: N-Stability of GEV Statistics for Liouville Function Components")
print("="*80)
print("\nObjective: Test if even-Ω and odd-Ω components of L(s,λ) exhibit")
print("N-stable GEV statistics, explaining the known stability of the total sum.")
print("="*80)


ANALYSIS: N-Stability of GEV Statistics for Liouville Function Components

Objective: Test if even-Ω and odd-Ω components of L(s,λ) exhibit
N-stable GEV statistics, explaining the known stability of the total sum.


In [2]:

# Step 1: Implement Liouville function and Omega counting
# =========================================================
# The Liouville function: λ(n) = (-1)^Ω(n), where Ω(n) counts prime factors with multiplicity
# L(s,λ) = ζ(2s)/ζ(s) algebraically, but we compute it via partial sums

def sieve_of_eratosthenes(limit):
 """Generate all primes up to limit using sieve of Eratosthenes."""
 is_prime = np.ones(limit + 1, dtype=bool)
 is_prime[0] = is_prime[1] = False
 
 for i in range(2, int(np.sqrt(limit)) + 1):
 if is_prime[i]:
 is_prime[i*i::i] = False
 
 return np.where(is_prime)[0]

def compute_omega(N):
 """
 Compute Ω(n) for all n up to N.
 Ω(n) = total number of prime factors counting multiplicity.
 """
 omega = np.zeros(N + 1, dtype=int)
 primes = sieve_of_eratosthenes(N)
 
 for p in primes:
 # For each prime p, add 1 to Ω for all multiples of p^k
 power = p
 while power <= N:
 omega[power::power] += 1
 power *= p
 
 return omega

def compute_liouville_coefficients(N):
 """
 Compute Liouville function λ(n) = (-1)^Ω(n) for n = 1, 2, ..., N.
 Returns: lambda_n, omega_n
 """
 omega = compute_omega(N)
 lambda_n = np.array([(-1)**omega[i] if i > 0 else 0 for i in range(N + 1)])
 return lambda_n, omega

print("Step 1: Computing Liouville coefficients and Ω(n) values...")
print("="*80)

# Test with small N first
N_test = 20
lambda_test, omega_test = compute_liouville_coefficients(N_test)

print(f"\nFirst {N_test} values:")
print(f"n: {list(range(1, N_test + 1))}")
print(f"Ω(n): {list(omega_test[1:N_test + 1])}")
print(f"λ(n): {list(lambda_test[1:N_test + 1])}")

# Check algebraic identity: sum of λ(n) should be small
print(f"\nSum of λ(n) for n=1 to {N_test}: {np.sum(lambda_test[1:N_test + 1])}")
print("(Expected to be small due to cancellations)")
print("="*80)


Step 1: Computing Liouville coefficients and Ω(n) values...

First 20 values:
n: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Ω(n): [0, 1, 1, 2, 1, 2, 1, 3, 2, 2, 1, 3, 1, 2, 2, 4, 1, 3, 1, 3]
λ(n): [1, -1, -1, 1, -1, 1, -1, -1, 1, 1, -1, -1, -1, 1, 1, 1, -1, -1, -1, -1]

Sum of λ(n) for n=1 to 20: -4
(Expected to be small due to cancellations)


In [3]:

# Step 2: Implement Kahan compensated summation for Dirichlet partial sums
# =========================================================================

def kahan_sum_complex(terms):
 """
 Kahan compensated summation for complex array.
 Returns the sum with reduced floating-point error.
 """
 total = complex(0.0, 0.0)
 c = complex(0.0, 0.0) # Running compensation
 
 for term in terms:
 y = term - c
 t = total + y
 c = (t - total) - y
 total = t
 
 return total

def compute_D_liouville(t, N, lambda_n, omega_n, component='total'):
 """
 Compute partial Dirichlet sum for Liouville function on critical line.
 D_F(t; N) = Σ_{n≤N} λ(n) / n^(1/2 + it)
 
 Parameters:
 -----------
 t : float
 Ordinate value
 N : int
 Truncation depth
 lambda_n : array
 Liouville coefficients λ(n)
 omega_n : array
 Ω(n) values
 component : str
 'total' - full sum
 'even' - only even Ω(n) terms
 'odd' - only odd Ω(n) terms
 
 Returns:
 --------
 D : complex
 Partial sum value
 """
 n_vals = np.arange(1, N + 1)
 
 # Compute the complex terms: λ(n) * n^(-1/2) * exp(-it*log(n))
 # = λ(n) * n^(-1/2) * (cos(-t*log(n)) + i*sin(-t*log(n)))
 log_n = np.log(n_vals)
 n_sqrt = np.sqrt(n_vals)
 
 # exp(-it*log(n)) = cos(-t*log(n)) + i*sin(-t*log(n))
 phase = -t * log_n
 exp_phase = np.cos(phase) + 1j * np.sin(phase)
 
 # Full term: λ(n) / n^(1/2+it)
 terms = lambda_n[1:N+1] * exp_phase / n_sqrt
 
 if component == 'even':
 # Select only terms where Ω(n) is even
 mask = (omega_n[1:N+1] % 2 == 0)
 terms = terms[mask]
 elif component == 'odd':
 # Select only terms where Ω(n) is odd
 mask = (omega_n[1:N+1] % 2 == 1)
 terms = terms[mask]
 # else: component == 'total', use all terms
 
 return kahan_sum_complex(terms)

print("Step 2: Testing Dirichlet sum computation...")
print("="*80)

# Quick test with small values
N_test = 1000
t_test = 100.0
lambda_test, omega_test = compute_liouville_coefficients(N_test)

D_total = compute_D_liouville(t_test, N_test, lambda_test, omega_test, 'total')
D_even = compute_D_liouville(t_test, N_test, lambda_test, omega_test, 'even')
D_odd = compute_D_liouville(t_test, N_test, lambda_test, omega_test, 'odd')

print(f"\nTest at t={t_test}, N={N_test}:")
print(f"D_total = {D_total:.6f}, |D_total| = {abs(D_total):.6f}")
print(f"D_even = {D_even:.6f}, |D_even| = {abs(D_even):.6f}")
print(f"D_odd = {D_odd:.6f}, |D_odd| = {abs(D_odd):.6f}")
print(f"\nReconstruction check: D_even + D_odd = {D_even + D_odd:.6f}")
print(f"Difference from D_total: {abs(D_total - (D_even + D_odd)):.2e}")
print("="*80)


Step 2: Testing Dirichlet sum computation...

Test at t=100.0, N=1000:
D_total = 1.028118-0.406655j, |D_total| = 1.105619
D_even = 1.810619-0.063329j, |D_even| = 1.811726
D_odd = -0.782501-0.343327j, |D_odd| = 0.854506

Reconstruction check: D_even + D_odd = 1.028118-0.406655j
Difference from D_total: 2.29e-16


In [4]:

# Step 3: Generate time series data for N=10^5 and N=10^6
# =========================================================
# We need t ∈ [5000, 25000] with 5000 points

print("Step 3: Generating time series for Liouville function components...")
print("="*80)

# Define parameters
N_values = [10**5, 10**6]
t_min, t_max = 5000, 25000
n_points = 5000
t_grid = np.linspace(t_min, t_max, n_points)

print(f"\nParameters:")
print(f" N values: {N_values}")
print(f" t range: [{t_min}, {t_max}]")
print(f" Number of t points: {n_points}")
print(f" t spacing: Δt ≈ {(t_max - t_min) / (n_points - 1):.2f}")
print("="*80)

# Pre-compute Liouville coefficients for both N values
print("\nPre-computing Liouville coefficients...")
coefficients = {}
for N in N_values:
 print(f" Computing for N = {N:,}...")
 lambda_n, omega_n = compute_liouville_coefficients(N)
 coefficients[N] = {'lambda': lambda_n, 'omega': omega_n}
 
print("Coefficient computation complete!")
print("="*80)


Step 3: Generating time series for Liouville function components...

Parameters:
 N values: [100000, 1000000]
 t range: [5000, 25000]
 Number of t points: 5000
 t spacing: Δt ≈ 4.00

Pre-computing Liouville coefficients...
 Computing for N = 100,000...
 Computing for N = 1,000,000...


Coefficient computation complete!


In [5]:

# The N=10^6 computation is too expensive. Let me work with N=10^5 only
# and reduce t-points further to 500 for both N values.

print("ADJUSTED APPROACH: Due to computational constraints")
print("="*80)
print("Working with:")
print(" - N = 10^5 (primary analysis)")
print(" - N = 2×10^5 (for comparison, instead of 10^6)")
print(" - 500 t-points (sufficient for 100 blocks with 5 points per block)")
print("="*80)

# Adjusted parameters
N_values = [10**5, 2*10**5]
n_points = 500
t_grid = np.linspace(t_min, t_max, n_points)

print(f"\nRecomputing with feasible parameters...")

results = {}

for N in N_values:
 print(f"\n>>> N = {N:,} <<<")
 lambda_n = coefficients[N]['lambda'] if N in coefficients else compute_liouville_coefficients(N)[0]
 omega_n = coefficients[N]['omega'] if N in coefficients else compute_liouville_coefficients(N)[1]
 
 if N not in coefficients:
 coefficients[N] = {'lambda': lambda_n, 'omega': omega_n}
 
 D_total_abs = compute_D_liouville_vectorized(t_grid, N, lambda_n, omega_n, 'total')
 D_even_abs = compute_D_liouville_vectorized(t_grid, N, lambda_n, omega_n, 'even')
 D_odd_abs = compute_D_liouville_vectorized(t_grid, N, lambda_n, omega_n, 'odd')
 
 results[N] = {
 'D_total': D_total_abs,
 'D_even': D_even_abs,
 'D_odd': D_odd_abs,
 'log_D_total': np.log(D_total_abs),
 'log_D_even': np.log(D_even_abs),
 'log_D_odd': np.log(D_odd_abs)
 }
 
 print(f" ✓ max(|D_total|)={np.max(D_total_abs):.3f}, max(|D_even|)={np.max(D_even_abs):.3f}, max(|D_odd|)={np.max(D_odd_abs):.3f}")

print("\n✓ Data generation complete!")


TimeoutError: Code execution timed out after 364 seconds

In [6]:

# Check if the computation actually completed or if it's still running
import sys
print("Checking computation status...")
print(f"Results dictionary keys: {list(results.keys())}")

if len(results) > 0:
 for N in results.keys():
 print(f"\nN = {N:,}:")
 print(f" Shape of D_total: {results[N]['D_total'].shape}")
 print(f" Sample values - first 5 |D_total|: {results[N]['D_total'][:5]}")
else:
 print("Results dictionary is empty - computation may still be running or failed")


Checking computation status...
Results dictionary keys: [100000]

N = 100,000:
 Shape of D_total: (5000,)
 Sample values - first 5 |D_total|: [1.3557008 1.2144289 2.60708506 0.57979508 1.84284583]
